This script just combines the partial data files generated when using SLURM job arrays. It is not necessary to run this script if you are not using SLURM job arrays.

In [4]:
import os
import glob
import h5py
import numpy as np
import re
from tqdm.auto import tqdm

def extract_number(filename):
    # Extracts the number from a filename
    matches = re.findall(r"\d+", filename)
    if matches:
        return int(matches[-2])  # Consider the last numerical value for ordering
    else:
        return None

def combine_data(directory, init_name, finale_name, ext, remove_files=True):
    # Get a list of all h5py files that match the pattern, i.e., end with a SLURM job array index
    file_pattern = os.path.join(directory, init_name+"_[0-9]*"+ext)
    files_to_combine = glob.glob(file_pattern)
    files_to_combine = sorted(files_to_combine, key=extract_number)

    # Create a new h5py file to hold all the combined data
    with h5py.File(directory + finale_name + ext + '.nc', 'w') as hf_combined:
        for file in tqdm(files_to_combine, desc='processing files'):
            with h5py.File(file, 'r') as hf:
                # For each key (dataset) in the file
                for key in hf.keys():
                    # If the dataset already exists in the combined file, append to it
                    if key in hf_combined:
                        hf_combined[key].resize((hf_combined[key].shape[0] + hf[key].shape[0],) + hf[key].shape[1:])
                        hf_combined[key][-hf[key].shape[0]:] = hf[key]
                    else:
                        # Else, copy the entire dataset to the combined file
                        hf_combined.create_dataset(key, data=hf[key], maxshape=(None,) + hf[key].shape[1:], compression="gzip")

    os.replace(directory + finale_name + ext + '.nc', directory + finale_name + ext)

    if remove_files:
        for file in tqdm(files_to_combine, desc='removing partial files'):
            os.remove(file)


/tmp/ipykernel_253623/429009157.py:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [5]:
base_name = '1024_T_5x10_fnl-1000-1000'

# Be sure to change this
final_name = '1024_T_1000x10_fnl-1000-1000'

cache_dir = 'data/ksw/alm_cache/'
data_dir = 'data/ksw/lensed/'

In [6]:
ext = '.alms.hdf5'
combine_data(cache_dir, base_name, final_name, ext)

processing files:   0%|          | 0/200 [00:00<?, ?it/s]

removing files:   0%|          | 0/200 [00:00<?, ?it/s]

In [7]:
ext = '.alms_ng.hdf5'
combine_data(cache_dir, base_name, final_name, ext)

processing files:   0%|          | 0/200 [00:00<?, ?it/s]

removing files:   0%|          | 0/200 [00:00<?, ?it/s]

In [8]:
ext = '.hdf5'
combine_data(data_dir, base_name, final_name, ext)

processing files:   0%|          | 0/200 [00:00<?, ?it/s]

removing files:   0%|          | 0/200 [00:00<?, ?it/s]